# Feature Engineering: Band Statistics, PCA, k-means, nearest neighbor background estimate

Owner: Madeleine. Audience: the team.

This notebook explains turning each 15 × 25 window into one row our models can use. It uses `window_features` to <>. It uses Spark for <> and pandas for <>.

**Prerequisites**
- This branch, `madeleine/features`.
- data/labeled_windows.npz, built by scripts/build_labeled_windows.py.
- Test.csv in the repo root, to check the layout. 

**Outline**
1.


## 1. Setup and data location

In [1]:
# avoid using stale version of script
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path.cwd()
if not (ROOT / "src" / "uvnc").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from uvnc.units import db_to_linear, linear_to_db 
from uvnc.windows import windows_from_frame 
from uvnc.features import window_features, FEATURE_NAMES


LABELED = ROOT / "data" / "labeled_windows.npz"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})



## 2. Labeled windows

In [3]:
data = np.load(LABELED)
print(data.files)

['mixtures', 'shares', 'site_ids']


In [4]:
#get arrays 
mixtures = data['mixtures']
shares = data['shares']
site_ids = data['site_ids']

for key in data.files:
    print(f"{key}: {data[key].shape}")

mixtures: (87752, 15, 25)
shares: (87752,)
site_ids: (87752,)


In [5]:
missing_minutes = np.isnan(mixtures).all(axis=2).sum(axis=1)
pd.Series(missing_minutes).value_counts().sort_index()

2    87752
Name: count, dtype: int64

In [6]:
display(np.isnan(mixtures[:, :, 0]).all(axis=0))

array([False, False, False, False, False, False, False, False, False,
        True, False,  True, False, False, False])

In [7]:
display(pd.Series(shares).describe())

count    8.775200e+04
mean     2.936509e-01
std      3.521028e-01
min      9.441511e-09
25%      1.084547e-02
50%      7.637237e-02
75%      6.082307e-01
max      9.998537e-01
dtype: float64

In [8]:
display(pd.Series(site_ids).value_counts())

1     52134
5     16698
13     8867
7      3371
4      3168
2      2976
9       294
10      244
Name: count, dtype: int64

## 3. Features

In [9]:

features = window_features(mixtures)
print(features.shape)
print(np.nanmin(features), np.nanmax(features))

(87752, 78)
0.0013248125652864928 74.67807574348848


In [10]:
print(len(FEATURE_NAMES))
print(FEATURE_NAMES[:3], FEATURE_NAMES[-3:])

78
['peak_minus_median_TOL_31.6', 'peak_minus_median_TOL_39.8', 'peak_minus_median_TOL_50.1'] ['broadband_peak_minus_median', 'broadband_peak_minute', 'missing_minutes']


In [11]:
features = window_features(mixtures)

print(features.shape, features.shape[1] == len(FEATURE_NAMES))

print(pd.Series(features[:, 77]).value_counts())#missing min
print(pd.Series(features[:, 76]).value_counts())#peak min

(87752, 78) True
2.0    87752
Name: count, dtype: int64
8.0     28707
1.0     16400
9.0      6551
7.0      6237
15.0     4358
11.0     3766
6.0      3669
13.0     3115
14.0     3078
2.0      3035
5.0      3032
4.0      2918
3.0      2886
Name: count, dtype: int64


In [13]:
display(pd.Series(shares).groupby(features[:, 76]).median())

1.0     0.010086
2.0     0.021134
3.0     0.018408
4.0     0.020545
5.0     0.024860
6.0     0.041397
7.0     0.181031
8.0     0.784837
9.0     0.173033
11.0    0.030400
13.0    0.021715
14.0    0.019983
15.0    0.026297
dtype: float64